<a href="https://colab.research.google.com/github/marcio-antonio/NumericalModeling_wingPlane/blob/main/Project_numericalModeling_wingPlane.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
* ================================================================
* SAE AERODESIGN - LONGARINA
* STAGE 4B - FINITE ELEMENT MODEL
* INTERNAL FORCES, STRESSES, AND POST-PROCESSING
* ================================================================

OPTI DIME 3 ELEM SEG2 MODE TRID;

* ------------------------------------------------
* GEOMETRY
* ------------------------------------------------

L = 10.;
D = 0.20;

P1 = 0. 0. 0.;
P2 = L 0. 0.;

* 50 beam elements
BEAM = DROI 50 P1 P2;

* Aircraft weight location node (snapped to mesh)
PW = BEAM POIN 'PROC' (3. 0. 0.);

* ------------------------------------------------
* MATERIAL
* ------------------------------------------------

E = 210.E9;
NU = 0.30;
RHO = 7800.;

* ------------------------------------------------
* SECTION
* ------------------------------------------------

PI = 3.14159265359;
A = PI*D*D/4.;
I = PI*D*D*D*D/64.;
J = PI*D*D*D*D/32.;

C = D/2.;

* ------------------------------------------------
* LOADS
* ------------------------------------------------

Q = 6000.;

MASS = 250.;
G = 9.81;
W = MASS*G;

WNEG = 0. - W;

* ------------------------------------------------
* ANALYTICAL REFERENCE
* ------------------------------------------------

FQ = Q*L;

VROOT = FQ - W;

MOMQ = Q*L*L/2.;
MOMW = W*3.;
MROOT = MOMQ - MOMW;

SIGROOT = MROOT*C/I;
TAUROOT = 4.*VROOT/(3.*A);

MESS ' ';
MESS '================================================';
MESS 'ANALYTICAL REFERENCE';
MESS '================================================';
MESS 'AERODYNAMIC FORCE   = ' FQ;
MESS 'AIRCRAFT WEIGHT     = ' W;
MESS 'ROOT SHEAR          = ' VROOT;
MESS 'ROOT MOMENT         = ' MROOT;
MESS 'ROOT BENDING STRESS = ' SIGROOT;
MESS 'ROOT SHEAR STRESS   = ' TAUROOT;
MESS '================================================';

* ------------------------------------------------
* BEAM MODEL & STIFFNESS
* ------------------------------------------------

MOD = MODE BEAM MECANIQUE ELASTIQUE POUT;

VY = 0. 1. 0.;

MAT = MATE MOD
      'YOUN' E
      'NU'   NU
      'RHO'  RHO
      'SECT' A
      'INRY' I
      'INRZ' I
      'TORS' J
      'VECT' VY;

RIG = RIGI MOD MAT;

* Root clamp
BLOC = BLOQ 'DEPL' 'ROTA' P1;
RIGB = RIG ET BLOC;

* ------------------------------------------------
* APPLIED LOADS
* ------------------------------------------------

MODB = MODE BEAM MECANIQUE ELASTIQUE BARR;
MATB = MATE MODB 'YOUN' 1. 'SECT' 1. 'RHO' 1.;
MASB = MASSE MODB MATB;

LOADY = MANU 'CHPO' BEAM 1 'UY' Q;
LOADQ = MASB * LOADY;

LOADW = FORC 'FY' WNEG PW;

LOAD = LOADQ ET LOADW;

* ------------------------------------------------
* SOLVE
* ------------------------------------------------

U = RESO RIGB LOAD;

* ------------------------------------------------
* REACTIONS & VERIFICATION
* ------------------------------------------------

REAC_FE = REAC BLOC U;

FY_FE = EXTR REAC_FE 'FY' P1;
MZ_FE = EXTR REAC_FE 'MZ' P1;

FY_FE_ABS = ABS FY_FE;
MZ_FE_ABS = ABS MZ_FE;

DIFF_FY = ABS (FY_FE_ABS - VROOT);
ERR_FY  = 100. * DIFF_FY / VROOT;

DIFF_MZ = ABS (MZ_FE_ABS - MROOT);
ERR_MZ  = 100. * DIFF_MZ / MROOT;

MESS ' ';
MESS '================================================';
MESS 'FE REACTION RESULTS & COMPARISON';
MESS '================================================';
MESS 'FE ROOT REACTION FY   = ' FY_FE_ABS;
MESS 'FE ROOT REACTION MZ   = ' MZ_FE_ABS;
MESS 'FY ERROR (%)          = ' ERR_FY;
MESS 'MZ ERROR (%)          = ' ERR_MZ;
MESS '================================================';

* ------------------------------------------------
* 1. INTERNAL EFFORTS (SHEAR V & BENDING MOMENT M)
* ------------------------------------------------

* Internal force field
SIG = SIGM MOD MAT U;

* Convert MCHAML to CHPOINT at nodes
CHPO_SIG = CHAN 'CHPO' MOD SIG;

* Extract values at Root (P1)
VY_ROOT_FE = EXTR CHPO_SIG 'EFFY' P1;
MZ_ROOT_FE = EXTR CHPO_SIG 'MOMZ' P1;

* Isolate components using EXCO
CHPO_VY = EXCO 'EFFY' CHPO_SIG;
CHPO_MZ = EXCO 'MOMZ' CHPO_SIG;

* Find maximum scalar values along the spar
VY_MAX_FE = MAXI CHPO_VY;
MZ_MAX_FE = MAXI CHPO_MZ;

* Generate 1D evolution curves along X axis
EV_VY = EVOL 'CHPO' CHPO_SIG 'EFFY' BEAM;
EV_MZ = EVOL 'CHPO' CHPO_SIG 'MOMZ' BEAM;

MESS ' ';
MESS '================================================';
MESS 'INTERNAL EFFORTS SUMMARY';
MESS '================================================';
MESS 'FE ROOT SHEAR  (EFFY at P1)    = ' VY_ROOT_FE;
MESS 'FE ROOT MOMENT (MOMZ at P1)    = ' MZ_ROOT_FE;
MESS 'FE MAX SHEAR ALONG SPAR        = ' VY_MAX_FE;
MESS 'FE MAX BENDING MOMENT          = ' MZ_MAX_FE;
MESS '================================================';

* ------------------------------------------------
* 2. BENDING STRESS FIELD
* ------------------------------------------------

FACTOR = C / I;

* Calculate root bending stress directly from scalar moment
SIG_ROOT_FE = ABS (MZ_ROOT_FE * FACTOR);

* Calculate max stress along spar via CHPOINT
CHPO_STRESS = CHPO_MZ * FACTOR;
SIG_MAX_FE  = ABS (MAXI CHPO_STRESS);

DIFF_SIG = ABS (SIG_ROOT_FE - SIGROOT);
ERR_SIG  = 100. * DIFF_SIG / SIGROOT;

* Evolution curve of stress along X
EV_SIG = EVOL 'CHPO' CHPO_STRESS 'SCAL' BEAM;

MESS ' ';
MESS '================================================';
MESS 'STRESS RESULTS SUMMARY';
MESS '================================================';
MESS 'ROOT BENDING STRESS FE (Pa)  = ' SIG_ROOT_FE;
MESS 'MAX SPAR BENDING STRESS (Pa)  = ' SIG_MAX_FE;
MESS 'ANALYTICAL ROOT BENDING (Pa)  = ' SIGROOT;
MESS 'ROOT STRESS ERROR (%)         = ' ERR_SIG;
MESS '================================================';

* ------------------------------------------------
* 3. DEFORMED SHAPE & DISPLACEMENT
* ------------------------------------------------

UY_TIP = EXTR U 'UY' P2;

MESS ' ';
MESS '================================================';
MESS 'DISPLACEMENT SUMMARY';
MESS '================================================';
MESS 'TIP VERTICAL DISPLACEMENT (m)  = ' UY_TIP;
MESS '================================================';

DEF0 = DEFO BEAM U 0.;
DEF1 = DEFO BEAM U 1. 'ROUG';

* ================================================================
* COMPLETE GRAPHICAL VISUALIZATION BLOCK
* ================================================================

* Option A: Interactive Screen Windows (Default)
OPTI 'TRAC' 'X' ;

* 1. EVOL CURVES (2D Plots)
LEG1 = MOTS 'V(x)' ;
LEG2 = MOTS 'M(x)' ;
LEG3 = MOTS 'Sigma(x)' ;

DESS EV_VY  'TITR' 'Shear Force Diagram V(x) [N]' 'LEGE' LEG1 ;
DESS EV_MZ  'TITR' 'Bending Moment Diagram M(x) [N.m]' 'LEGE' LEG2 ;
DESS EV_SIG 'TITR' 'Bending Stress Diagram Sigma(x) [Pa]' 'LEGE' LEG3 ;

* 2. STRUCTURAL PLOTS (Deformed Shape & Field Contours)
DEF0 = DEFO BEAM U 0. ;
DEF1 = DEFO BEAM U 1. 'ROUG' ;

* ------------------------------------------------
* 2. STRUCTURAL PLOTS (WITH DIRECT FRONT VIEW)
* ------------------------------------------------

* Set camera viewpoint centered at mid-span (X=5, Y=0, Z=10)
EYE = 5. 0. 10. ;
OPTI 'OEIL' EYE ;

* Define initial mesh (DEF0) and deformed mesh (DEF1)
DEF0 = DEFO BEAM U 0. ;
DEF1 = DEFO BEAM U 1. 'ROUG' ;

* 1. Deformed shape vs initial spar
TRAC (DEF0 ET DEF1) 'TITR' 'Deformed Shape vs Initial Spar' ;

* 2. Vertical displacement contour map
UY_FIELD = EXCO 'UY' U ;
TRAC UY_FIELD BEAM 'TITR' 'Vertical Displacement Contour Uy [m]' ;

* 3. Bending stress contour map
TRAC CHPO_STRESS BEAM 'TITR' 'Bending Stress Field Sigma(x) [Pa]' ;

FIN;